In [1]:
import os
import xarray as xr
import numpy as np
from pathlib import Path
from glob import glob
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import logger

import matplotlib.pyplot as plt
import cartopy.crs as ccrs

from dask.distributed import Client, as_completed

ERROR 1: PROJ: proj_create_from_database: Open of /g/data/hh5/public/apps/miniconda3/envs/analysis3-24.04/share/proj failed


In [2]:
LOG = logger.get_logger(__name__)

In [3]:
client = Client(
    n_workers=24,
    threads_per_worker=1
)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /proxy/8787/status,
Dashboard: /proxy/8787/status,Workers: 24
Total threads: 24,Total memory: 95.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:44871,Workers: 24
Dashboard: /proxy/8787/status,Total threads: 24
Started: Just now,Total memory: 95.00 GiB
Comm: tcp://127.0.0.1:43655,Total threads: 1
Dashboard: /proxy/42511/status,Memory: 3.96 GiB
Nanny: tcp://127.0.0.1:41369,


In [4]:
def mean_daily(date):
    
    month = date[0:2]
    year = date[3:7]
    
    ##### ERA5 Data
    directory=Path(f"/g/data/rt52/era5/single-levels/reanalysis/msdwswrf/{year}")
    file = [f for f in directory.rglob(f"*{year}{month}*.nc")][0]

    ds = xr.open_dataset(file, chunks='auto')

    # convert hourly irradiance to daily insolation
    daily = ds.msdwswrf.resample(time='1D').sum()
    daily = daily * 0.0036 # MJ/m2
    
    return daily.mean(dim='time')

In [5]:
if __name__ == '__main__':
    for month in range(1,13):
        futures = {}
        for year in range(2016, 2023):
            date = f'{month:02d}-{year}'
            future = client.submit(mean_daily, date)
            futures[future] = date
    
        results = []
        labels = []
        for future in as_completed(futures):
            result = future.result()
            results.append(result)
            labels.append(futures[future])
    
        combined = xr.concat(results, dim='date')
        combined = combined.assign_coords(date=labels)
    
        mean = combined.mean(dim='date', skipna=True)
        file_path = Path('/g/data/er8/users/cd3022/solar_drought/era5_irradiance_means/')
        file_name = f'{month:02d}_month_mean.nc'
        os.makedirs(file_path, exist_ok=True)
        mean.to_netcdf(f'{file_path}/{file_name}')

ERROR 1: PROJ: proj_create_from_database: Open of /g/data/hh5/public/apps/miniconda3/envs/analysis3-24.04/share/proj failed
ERROR 1: PROJ: proj_create_from_database: Open of /g/data/hh5/public/apps/miniconda3/envs/analysis3-24.04/share/proj failed
ERROR 1: PROJ: proj_create_from_database: Open of /g/data/hh5/public/apps/miniconda3/envs/analysis3-24.04/share/proj failed
ERROR 1: PROJ: proj_create_from_database: Open of /g/data/hh5/public/apps/miniconda3/envs/analysis3-24.04/share/proj failed
ERROR 1: PROJ: proj_create_from_database: Open of /g/data/hh5/public/apps/miniconda3/envs/analysis3-24.04/share/proj failed
ERROR 1: PROJ: proj_create_from_database: Open of /g/data/hh5/public/apps/miniconda3/envs/analysis3-24.04/share/proj failed
ERROR 1: PROJ: proj_create_from_database: Open of /g/data/hh5/public/apps/miniconda3/envs/analysis3-24.04/share/proj failed
INFO:flox:Entering _validate_reindex: reindex is None
INFO:flox:Leaving _validate_reindex: method = None, returning None
INFO:flox:_